<a href="https://colab.research.google.com/github/Akash-hasnat/Leakage-Aware-Explainable-Stacking-Ensembles-for-Cross-Cohort-Cardiovascular-Risk-Prediction/blob/main/CVD_Cross_Cohort_Analysis_Colab_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Leakage-Aware, Explainable Stacking Ensembles for Cross-Cohort Cardiovascular Risk Prediction

**End-to-end reproducible pipeline** for the paper *"Cross-Cohort Generalization of a Leakage-Aware Stacking Ensemble for Cardiovascular Risk Prediction: A SHAP–LIME–ELI5 Explanation-Agreement Study."*

**Datasets** (place in the same folder as this notebook):
- `cardio_data/cardio_train.csv` — Kaggle Cardiovascular Disease dataset (70,000 records) — **training / internal validation**
- `framingham_data/framingham.csv` — Framingham Heart Study dataset (4,240 records) — **external validation only**

**Pipeline stages** (one section per stage, matching the paper's Methods subsections):
1. Feature harmonization across cohorts + two-level deduplication
2. Deduplication-before-split leakage audit + naive vs. leakage-free preprocessing comparison (impute/scale probe + SMOTE probe)
3. Base learners (Logistic Regression, Random Forest, XGBoost, Linear SVM) + Stacking ensemble; internal + external evaluation
4. Ablation study (deduplication, ensemble composition, feature groups)
5. Explainable AI: SHAP, LIME, ELI5 permutation importance + Kendall's τ explanation agreement
6. Statistical significance (McNemar's test) + reliability analysis (bootstrap CIs, ECE, CV stability)
7. Figure generation for the paper

**Requirements**: `pandas numpy scikit-learn xgboost imbalanced-learn shap lime eli5 statsmodels matplotlib`

```bash
pip install pandas numpy scikit-learn xgboost imbalanced-learn shap lime eli5 statsmodels matplotlib
```

All intermediate artifacts (harmonized data, model performance tables, ablation results, XAI rankings, statistical test results, figures) are written to `outputs/`.


> **This notebook is set up to run directly in Google Colab.** Runtime → Run all
> will install dependencies, ask you to upload the two dataset CSVs (or pull them
> from Google Drive), run the full pipeline, and let you download the results
> (tables, figures, and a zipped `outputs/` folder) at the end.
>
> Recommended runtime: **CPU is fine** (nothing here requires a GPU); the full run
> takes roughly 10–15 minutes, mostly in the ablation and LIME/XAI cells.

In [ ]:
# ============================================================
# 1. Install dependencies (Colab ships pandas/numpy/scikit-learn/matplotlib
#    already, but pin/add the packages this pipeline specifically needs)
# ============================================================
!pip install -q xgboost imbalanced-learn shap lime eli5 statsmodels

import sys
print("Python:", sys.version.split()[0])
import pandas, numpy, sklearn, xgboost, shap, lime, eli5, statsmodels
for pkg in [pandas, numpy, sklearn, xgboost, shap, eli5, statsmodels]:
    print(f"{pkg.__name__:15s} {pkg.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 6.5 MB/s eta 0:00:00
Python: 3.12.13
pandas          2.2.3
numpy           2.0.2
sklearn         1.6.1
xgboost         3.4.0
shap            0.52.0
eli5            0.16.0
statsmodels     0.14.6


## Get the data

Run **one** of the two cells below.

- **Option A — upload directly** (simplest): choose `cardio_train.csv` and
  `framingham.csv` from your computer when prompted.
- **Option B — Google Drive**: if you've already uploaded the two CSVs to your
  Drive, mount it and point the paths at your files instead.

Either way, this notebook expects the files to end up at
`cardio_data/cardio_train.csv` and `framingham_data/framingham.csv` relative to
the Colab working directory — both option cells below take care of that.

In [ ]:
# ============================================================
# Option A: upload the two CSVs from your local machine
# (skip this cell if you're using Option B / Google Drive instead)
# ============================================================
import os
from google.colab import files

os.makedirs('cardio_data', exist_ok=True)
os.makedirs('framingham_data', exist_ok=True)

print("Select the Kaggle Cardiovascular Disease CSV (cardio_train.csv)...")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, 'cardio_data/cardio_train.csv')

print("\nSelect the Framingham Heart Study CSV (framingham.csv)...")
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, 'framingham_data/framingham.csv')

print("\nDone. Files in place:")
!ls -la cardio_data framingham_data

Select the Kaggle Cardiovascular Disease CSV (cardio_train.csv)...


Saving cardio.zip to cardio.zip

Select the Framingham Heart Study CSV (framingham.csv)...


Saving Framingham heart.zip to Framingham heart.zip

Done. Files in place:
cardio_data:
total 752
drwxr-xr-x 2 root root   4096 Aug 21 11:42 .
drwxr-xr-x 1 root root   4096 Aug 21 11:43 ..
-rw-r--r-- 1 root root 760066 Aug 21 11:42 cardio_train.csv

framingham_data:
total 68
drwxr-xr-x 2 root root  4096 Aug 21 11:43 .
drwxr-xr-x 1 root root  4096 Aug 21 11:43 ..
-rw-r--r-- 1 root root 59440 Aug 21 11:43 framingham.csv


In [ ]:
# ============================================================
# Option B: pull the two CSVs from Google Drive instead of uploading
# (skip this cell if you already ran Option A above)
# ============================================================
import os, shutil
from google.colab import drive
import zipfile

drive.mount('/content/drive')

# EDIT these two paths to point at your files in Drive:
DRIVE_CARDIO_ZIP = '/content/drive/MyDrive/cardio.zip'
DRIVE_FRAMINGHAM_ZIP = '/content/drive/MyDrive/Framingham heart.zip'

os.makedirs('cardio_data', exist_ok=True)
os.makedirs('framingham_data', exist_ok=True)

# Copy and extract cardio.zip
shutil.copy(DRIVE_CARDIO_ZIP, 'cardio_data/cardio.zip')
with zipfile.ZipFile('cardio_data/cardio.zip', 'r') as zip_ref:
    zip_ref.extractall('cardio_data')

# Copy and extract Framingham heart.zip
shutil.copy(DRIVE_FRAMINGHAM_ZIP, 'framingham_data/framingham.zip')
with zipfile.ZipFile('framingham_data/framingham.zip', 'r') as zip_ref:
    zip_ref.extractall('framingham_data')

print("Done. Files in place:")
!ls -la cardio_data framingham_data

Mounted at /content/drive
Done. Files in place:
cardio_data:
total 3628
drwxr-xr-x 2 root root    4096 Aug 21 11:43 .
drwxr-xr-x 1 root root    4096 Aug 21 11:43 ..
-rw-r--r-- 1 root root 2941524 Aug 21 11:43 cardio_train.csv
-rw------- 1 root root  760066 Aug 21 11:43 cardio.zip

framingham_data:
total 256
drwxr-xr-x 2 root root   4096 Aug 21 11:43 .
drwxr-xr-x 1 root root   4096 Aug 21 11:43 ..
-rw-r--r-- 1 root root 191803 Aug 21 11:43 framingham.csv
-rw------- 1 root root  59440 Aug 21 11:43 framingham.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Quick sanity check before running the pipeline
import pandas as pd
c = pd.read_csv('cardio_data/cardio_train.csv', sep=None, engine='python', nrows=5)
f = pd.read_csv('framingham_data/framingham.csv', nrows=5)
print("cardio_train.csv columns:", list(c.columns))
print("framingham.csv columns:", list(f.columns))
print("\nLooks good — proceed to Section 1 below.")

cardio_train.csv columns: ['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']
framingham.csv columns: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']

Looks good — proceed to Section 1 below.


## Setup

Creates the output directory structure. Run this first.

In [ ]:
import os
os.makedirs('outputs/tables', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
print("Output directories ready.")

Output directories ready.


## 1. Feature Harmonization Across Cohorts + Deduplication

Loads both raw datasets, deduplicates the source (cardio) cohort at the raw-record level, removes
physiologically implausible rows, maps both cohorts onto a shared eight-feature clinical schema, and
performs a **second** deduplication pass on the harmonized representation itself (since coarser
categorical features can cause distinct raw records to collapse onto identical modelling rows).
All deduplication happens strictly **before** any train/test split.

In [ ]:
import pandas as pd, numpy as np, json, os

OUT = 'outputs'
os.makedirs(f'{OUT}/tables', exist_ok=True)

# ---------------- Load ----------------
cardio = pd.read_csv('cardio_data/cardio_train.csv', sep=None, engine='python')
fram = pd.read_csv('framingham_data/framingham.csv')

log = {}
log['cardio_raw_shape'] = cardio.shape
log['framingham_raw_shape'] = fram.shape

# ---------------- Deduplication (BEFORE split) ----------------
feat_cols_no_id = [c for c in cardio.columns if c != 'id']
n_before = len(cardio)
dup_mask = cardio.duplicated(subset=feat_cols_no_id, keep='first')
n_dupes = int(dup_mask.sum())
cardio_dedup = cardio.loc[~dup_mask].reset_index(drop=True)
n_after = len(cardio_dedup)
log['cardio_duplicates_removed'] = n_dupes
log['cardio_shape_after_dedup'] = cardio_dedup.shape

n_before_f = len(fram)
dup_mask_f = fram.duplicated(keep='first')
fram_dedup = fram.loc[~dup_mask_f].reset_index(drop=True)
log['framingham_duplicates_removed'] = int(dup_mask_f.sum())
log['framingham_shape_after_dedup'] = fram_dedup.shape

# Also flag physiologically-implausible rows in cardio (documented data-quality step, not leakage-related)
c = cardio_dedup.copy()
implausible = (c['ap_hi'] <= 0) | (c['ap_lo'] <= 0) | (c['ap_hi'] < c['ap_lo']) | \
              (c['ap_hi'] > 250) | (c['ap_lo'] > 200) | (c['height'] < 100) | (c['height'] > 220) | \
              (c['weight'] < 30) | (c['weight'] > 200)
log['cardio_implausible_rows_removed'] = int(implausible.sum())
c = c.loc[~implausible].reset_index(drop=True)
log['cardio_shape_after_cleaning'] = c.shape

# ---------------- Harmonize CARDIO -> common schema ----------------
h1 = pd.DataFrame()
h1['age_years'] = (c['age'] / 365.25)
h1['sex_male'] = (c['gender'] == 2).astype(int)  # cardio: 1=female,2=male (per dataset docs)
h1['bmi'] = c['weight'] / ((c['height'] / 100.0) ** 2)
h1['sys_bp'] = c['ap_hi'].astype(float)
h1['dia_bp'] = c['ap_lo'].astype(float)
h1['smoker'] = c['smoke'].astype(int)
h1['chol_cat'] = c['cholesterol'].astype(int)          # 1 normal, 2 above normal, 3 well above normal
h1['gluc_cat'] = c['gluc'].astype(int)                 # 1 normal, 2 above normal, 3 well above normal
h1['alcohol'] = c['alco'].astype(int)
h1['active'] = c['active'].astype(int)
h1['target'] = c['cardio'].astype(int)
h1['source'] = 'cardio'

# ---------------- Harmonize FRAMINGHAM -> common schema ----------------
f = fram_dedup.copy()
h2 = pd.DataFrame()
h2['age_years'] = f['age'].astype(float)
h2['sex_male'] = f['male'].astype(int)
h2['bmi'] = f['BMI'].astype(float)
h2['sys_bp'] = f['sysBP'].astype(float)
h2['dia_bp'] = f['diaBP'].astype(float)
h2['smoker'] = f['currentSmoker'].astype(int)

def chol_bin(x):
    if pd.isna(x): return np.nan
    if x < 200: return 1
    if x < 240: return 2
    return 3
def gluc_bin(x):
    if pd.isna(x): return np.nan
    if x < 100: return 1
    if x < 126: return 2
    return 3

h2['chol_cat'] = f['totChol'].apply(chol_bin)
h2['gluc_cat'] = f['glucose'].apply(gluc_bin)
h2['alcohol'] = np.nan       # not available in Framingham -> will be imputed / documented as missing-domain feature
h2['active'] = np.nan        # not available in Framingham
h2['target'] = f['TenYearCHD'].astype(int)
h2['source'] = 'framingham'

CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

# Second-pass dedup at the harmonized/core-feature level: because coarser common
# features (e.g. clinical categories rather than raw glucose mg/dL) can map several
# distinct raw records onto identical modelling rows, we deduplicate again on the
# exact representation that will be fed to the models.
n_before_h1 = len(h1)
dup_mask_h1 = h1.duplicated(subset=CORE + ['target'], keep='first')
h1 = h1.loc[~dup_mask_h1].reset_index(drop=True)
log['cardio_harmonized_level_duplicates_removed'] = int(dup_mask_h1.sum())

log['harmonized_cardio_shape'] = h1.shape
log['harmonized_framingham_shape'] = h2.shape
log['common_features'] = [c for c in h1.columns if c not in ('target','source')]

h1.to_csv(f'{OUT}/tables/harmonized_cardio.csv', index=False)
h2.to_csv(f'{OUT}/tables/harmonized_framingham.csv', index=False)

with open(f'{OUT}/tables/harmonize_log.json','w') as fh:
    json.dump(log, fh, indent=2, default=str)

print(json.dumps(log, indent=2, default=str))

{
  "cardio_raw_shape": [
    70000,
    13
  ],
  "framingham_raw_shape": [
    4240,
    16
  ],
  "cardio_duplicates_removed": 24,
  "cardio_shape_after_dedup": [
    69976,
    13
  ],
  "framingham_duplicates_removed": 0,
  "framingham_shape_after_dedup": [
    4240,
    16
  ],
  "cardio_implausible_rows_removed": 1323,
  "cardio_shape_after_cleaning": [
    68653,
    13
  ],
  "cardio_harmonized_level_duplicates_removed": 6,
  "harmonized_cardio_shape": [
    68647,
    12
  ],
  "harmonized_framingham_shape": [
    4240,
    12
  ],
  "common_features": [
    "age_years",
    "sex_male",
    "bmi",
    "sys_bp",
    "dia_bp",
    "smoker",
    "chol_cat",
    "gluc_cat",
    "alcohol",
    "active"
  ]
}


## 2. Deduplication-Before-Split Audit + Naive vs. Leakage-Free Preprocessing

Two experiments:

- **Split-order audit**: splitting the *raw, non-deduplicated* cardio data first and counting how many
  test-set records have an identical duplicate already sitting in the training fold (i.e., how much
  leakage a naive split-before-dedup workflow would introduce).
- **Naive vs. leakage-free preprocessing**, evaluated two ways:
  1. Median imputation + standardization fit on the full dataset before 5-fold CV (naive) vs. fit only
     inside each training fold (leakage-free), probed with logistic regression.
  2. SMOTE oversampling applied before the train/test split (naive — the classic severe-leakage case)
     vs. applied only to the training fold after splitting (leakage-free), probed with a random forest.

The train/test split used for all downstream modelling (Sections 3–6) is also created here, from the
already-deduplicated harmonized cardio cohort.

In [ ]:
import pandas as pd, numpy as np, json, os
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

h1 = pd.read_csv(f'{OUT}/tables/harmonized_cardio.csv')   # already deduped+cleaned
raw_cardio = pd.read_csv('cardio_data/cardio_train.csv', sep=None, engine='python')

results = {}

# ---------------------------------------------------------------
# A) Illustrate WHY dedup-before-split matters: split RAW (with dupes)
#    and count how many duplicate-feature rows leak across train/test
# ---------------------------------------------------------------
feat_no_id = [c for c in raw_cardio.columns if c != 'id']
raw_tr, raw_te = train_test_split(raw_cardio, test_size=0.2, random_state=42, stratify=raw_cardio['cardio'])
tr_keys = set(map(tuple, raw_tr[feat_no_id].values.tolist()))
te_keys = list(map(tuple, raw_te[feat_no_id].values.tolist()))
leaked = sum(1 for k in te_keys if k in tr_keys)
results['naive_split_before_dedup'] = {
    'test_rows_with_identical_train_counterpart': leaked,
    'pct_of_test': round(100*leaked/len(raw_te), 3)
}

# Clean split: dedup BEFORE split (h1 already dedup+cleaned)
tr_idx, te_idx = train_test_split(h1.index, test_size=0.2, random_state=42, stratify=h1['target'])
train_df = h1.loc[tr_idx].reset_index(drop=True)
test_df  = h1.loc[te_idx].reset_index(drop=True)
train_df.to_csv(f'{OUT}/tables/cardio_train_split.csv', index=False)
test_df.to_csv(f'{OUT}/tables/cardio_test_split.csv', index=False)
results['clean_split_after_dedup'] = {'train_rows': len(train_df), 'test_rows': len(test_df)}

tr_keys2 = set(map(tuple, train_df[CORE].values.tolist()))
te_keys2 = list(map(tuple, test_df[CORE].values.tolist()))
leaked2 = sum(1 for k in te_keys2 if k in tr_keys2)
results['clean_split_after_dedup']['residual_identical_rows_in_test'] = leaked2

# ---------------------------------------------------------------
# B) Naive vs Leakage-free PREPROCESSING (imputation + scaling) comparison
#    Using 5-fold CV with Logistic Regression as the diagnostic probe
# ---------------------------------------------------------------
X = h1[CORE].values.astype(float)
y = h1['target'].values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# NAIVE: impute+scale on the FULL dataset BEFORE cross-validation (fold info leaks into transform)
imp_naive = SimpleImputer(strategy='median').fit(X)
X_imp = imp_naive.transform(X)
sc_naive = StandardScaler().fit(X_imp)
X_naive = sc_naive.transform(X_imp)
naive_scores = cross_val_score(LogisticRegression(max_iter=1000), X_naive, y, cv=skf, scoring='roc_auc')

# LEAKAGE-FREE: impute+scale INSIDE each CV fold (fit on train fold only) via Pipeline
pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
clean_scores = cross_val_score(pipe, X, y, cv=skf, scoring='roc_auc')

results['naive_vs_leakagefree_preprocessing_5fold_AUC'] = {
    'naive_mean_auc': float(np.mean(naive_scores)),
    'naive_std_auc': float(np.std(naive_scores)),
    'naive_fold_scores': naive_scores.tolist(),
    'leakage_free_mean_auc': float(np.mean(clean_scores)),
    'leakage_free_std_auc': float(np.std(clean_scores)),
    'leakage_free_fold_scores': clean_scores.tolist(),
    'optimistic_bias_auc': float(np.mean(naive_scores) - np.mean(clean_scores))
}

# ---------------------------------------------------------------
# C) Naive vs Leakage-free RESAMPLING (SMOTE) — the classic severe-leakage case:
#    oversampling the minority class BEFORE the train/test split lets synthetic
#    points be interpolated from neighbours that later land in the test fold,
#    and lets original minority test points be duplicated into the train fold.
# ---------------------------------------------------------------
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

def eval_rf(Xtr, ytr, Xte, yte):
    m = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, p)

# NAIVE: SMOTE on the full dataset first, then split the (partly synthetic) data
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
imp = SimpleImputer(strategy='median').fit(Xr_tr); sc = StandardScaler().fit(imp.transform(Xr_tr))
auc_naive_smote = eval_rf(sc.transform(imp.transform(Xr_tr)), yr_tr, sc.transform(imp.transform(Xr_te)), yr_te)

# LEAKAGE-FREE: split first, SMOTE fit/applied only on the training fold
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
imp2 = SimpleImputer(strategy='median').fit(Xtr); sc2 = StandardScaler().fit(imp2.transform(Xtr))
Xtr_s = sc2.transform(imp2.transform(Xtr)); Xte_s = sc2.transform(imp2.transform(Xte))
Xtr_res, ytr_res = SMOTE(random_state=42).fit_resample(Xtr_s, ytr)
auc_clean_smote = eval_rf(Xtr_res, ytr_res, Xte_s, yte)

results['naive_vs_leakagefree_SMOTE_resampling_AUC'] = {
    'naive_smote_before_split_auc': float(auc_naive_smote),
    'leakage_free_smote_after_split_auc': float(auc_clean_smote),
    'optimistic_bias_auc': float(auc_naive_smote - auc_clean_smote)
}

with open(f'{OUT}/tables/leakage_analysis.json','w') as f:
    json.dump(results, f, indent=2, default=str)
print(json.dumps(results, indent=2, default=str))

{
  "naive_split_before_dedup": {
    "test_rows_with_identical_train_counterpart": 3,
    "pct_of_test": 0.021
  },
  "clean_split_after_dedup": {
    "train_rows": 54917,
    "test_rows": 13730,
    "residual_identical_rows_in_test": 8
  },
  "naive_vs_leakagefree_preprocessing_5fold_AUC": {
    "naive_mean_auc": 0.7899465580811377,
    "naive_std_auc": 0.0036254509750713025,
    "naive_fold_scores": [
      0.7935961627720232,
      0.7914371087488485,
      0.783553446361009,
      0.7885089031651193,
      0.7926371693586882
    ],
    "leakage_free_mean_auc": 0.789946430734737,
    "leakage_free_std_auc": 0.0036252008978672084,
    "leakage_free_fold_scores": [
      0.7935953988168541,
      0.7914379575879253,
      0.7835538920671272,
      0.7885081815478648,
      0.7926367236539132
    ],
    "optimistic_bias_auc": 1.2734640064682168e-07
  },
  "naive_vs_leakagefree_SMOTE_resampling_AUC": {
    "naive_smote_before_split_auc": 0.7971385419202622,
    "leakage_free_smote_afte

## 3. Base Learners + Stacking Ensemble (Internal + External Evaluation)

Trains four base learners (Logistic Regression, Random Forest, XGBoost, Linear SVM with Platt-scaled
probabilities) and a stacking ensemble (5-fold out-of-fold predictions → logistic-regression
meta-learner) on the leakage-free-preprocessed cardio training split. All preprocessing (imputer,
scaler) is fit **only** on the training partition. Every model is then evaluated on both the internal
held-out cardio test set and the fully external, harmonized Framingham cohort — with **no refitting or
recalibration** on Framingham.

In [ ]:
import pandas as pd, numpy as np, json, os, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, precision_score,
                              recall_score, confusion_matrix, brier_score_loss)
import xgboost as xgb

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

train_df = pd.read_csv(f'{OUT}/tables/cardio_train_split.csv')
test_df  = pd.read_csv(f'{OUT}/tables/cardio_test_split.csv')
ext_df   = pd.read_csv(f'{OUT}/tables/harmonized_framingham.csv')

Xtr_raw, ytr = train_df[CORE].values.astype(float), train_df['target'].values
Xte_raw, yte = test_df[CORE].values.astype(float), test_df['target'].values
Xext_raw, yext = ext_df[CORE].values.astype(float), ext_df['target'].values

# ---- Leakage-free preprocessing: fit ONLY on training data ----
imputer = SimpleImputer(strategy='median').fit(Xtr_raw)
scaler = StandardScaler().fit(imputer.transform(Xtr_raw))

def prep(X):
    return scaler.transform(imputer.transform(X))

Xtr, Xte, Xext = prep(Xtr_raw), prep(Xte_raw), prep(Xext_raw)
joblib.dump({'imputer': imputer, 'scaler': scaler}, f'{OUT}/tables/preprocessor.joblib')

# ---- Base learners ----
base_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_leaf=5,
                                            random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
                                  random_state=42, n_jobs=-1),
    'LinearSVM': CalibratedClassifierCV(LinearSVC(max_iter=5000, random_state=42), cv=3)
}

def evaluate(model, X, y, name, split_name):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    sens = tp / (tp + fn); spec = tn / (tn + fp)
    return {
        'model': name, 'split': split_name,
        'auc': float(roc_auc_score(y, proba)),
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred)),
        'recall_sensitivity': float(sens),
        'specificity': float(spec),
        'f1': float(f1_score(y, pred)),
        'brier': float(brier_score_loss(y, proba)),
    }, proba, pred

results = []
fitted = {}
proba_store = {}   # for McNemar / agreement later
pred_store = {}

for name, mdl in base_models.items():
    mdl.fit(Xtr, ytr)
    fitted[name] = mdl
    r_int, p_int, pr_int = evaluate(mdl, Xte, yte, name, 'internal_test')
    r_ext, p_ext, pr_ext = evaluate(mdl, Xext, yext, name, 'external_framingham')
    results += [r_int, r_ext]
    proba_store[(name, 'internal_test')] = p_int; pred_store[(name, 'internal_test')] = pr_int
    proba_store[(name, 'external_framingham')] = p_ext; pred_store[(name, 'external_framingham')] = pr_ext
    print(name, 'internal AUC', round(r_int['auc'],4), 'external AUC', round(r_ext['auc'],4))

# ---- Stacking ensemble (meta-learner: Logistic Regression, 5-fold OOF stacking) ----
estimators = [(k, v) for k, v in base_models.items()]
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)
stack.fit(Xtr, ytr)
r_int, p_int, pr_int = evaluate(stack, Xte, yte, 'StackingEnsemble', 'internal_test')
r_ext, p_ext, pr_ext = evaluate(stack, Xext, yext, 'StackingEnsemble', 'external_framingham')
results += [r_int, r_ext]
proba_store[('StackingEnsemble', 'internal_test')] = p_int; pred_store[('StackingEnsemble', 'internal_test')] = pr_int
proba_store[('StackingEnsemble', 'external_framingham')] = p_ext; pred_store[('StackingEnsemble', 'external_framingham')] = pr_ext
print('Stacking internal AUC', round(r_int['auc'],4), 'external AUC', round(r_ext['auc'],4))

joblib.dump(fitted, f'{OUT}/tables/base_models.joblib')
joblib.dump(stack, f'{OUT}/tables/stack_model.joblib')

res_df = pd.DataFrame(results)
res_df.to_csv(f'{OUT}/tables/model_performance.csv', index=False)
print(res_df.to_string(index=False))

# save arrays needed downstream
np.savez(f'{OUT}/tables/pred_arrays.npz',
         yte=yte, yext=yext,
         **{f'proba__{k[0]}__{k[1]}': v for k, v in proba_store.items()},
         **{f'pred__{k[0]}__{k[1]}': v for k, v in pred_store.items()})

LogisticRegression internal AUC 0.7857 external AUC 0.6805
RandomForest internal AUC 0.7968 external AUC 0.6548
XGBoost internal AUC 0.7969 external AUC 0.6563
LinearSVM internal AUC 0.7856 external AUC 0.6812
Stacking internal AUC 0.7973 external AUC 0.6611
             model               split      auc  accuracy  precision  recall_sensitivity  specificity       f1    brier
LogisticRegression       internal_test 0.785660  0.721486   0.751866            0.652487     0.789072 0.698660 0.189134
LogisticRegression external_framingham 0.680505  0.516509   0.209023            0.784161     0.468576 0.330065 0.336435
      RandomForest       internal_test 0.796833  0.730517   0.757319            0.670150     0.789648 0.711073 0.182661
      RandomForest external_framingham 0.654795  0.462028   0.194703            0.810559     0.399611 0.313985 0.339686
           XGBoost       internal_test 0.796941  0.732848   0.752015            0.686488     0.778258 0.717759 0.182312
           XGBoost ex

## 4. Ablation Study

Three ablation axes, each re-training a stacking ensemble:

1. **Deduplication** — with vs. without pre-split duplicate removal.
2. **Stacking composition** — full four-learner stack vs. each base learner removed in turn.
3. **Feature groups** — full eight-feature set vs. each feature removed in turn.

All configurations are evaluated on both the internal test set and the external Framingham cohort.
A 20,000-record training subsample and lighter hyperparameters are used here (vs. Section 3's final
models) to keep the ~14-configuration grid computationally tractable; absolute AUCs differ slightly
from Section 3, but the *relative* deltas across rows are what matters for the ablation conclusions.

**Note:** this cell can take several minutes to run.

In [ ]:
import pandas as pd, numpy as np, json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

train_df_full = pd.read_csv(f'{OUT}/tables/cardio_train_split.csv')
test_df  = pd.read_csv(f'{OUT}/tables/cardio_test_split.csv')
ext_df   = pd.read_csv(f'{OUT}/tables/harmonized_framingham.csv')

# Subsample training data for the ablation grid to keep total runtime tractable
# (ablation is about *relative* deltas across ~13 configurations, not the final model)
train_df, _ = train_test_split(train_df_full, train_size=20000, random_state=42, stratify=train_df_full['target'])

def make_stack():
    base = [
        ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
        ('RandomForest', RandomForestClassifier(n_estimators=120, max_depth=8, min_samples_leaf=10, random_state=42, n_jobs=-1)),
        ('XGBoost', xgb.XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.08, subsample=0.8,
                                       colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)),
        ('LinearSVM', CalibratedClassifierCV(LinearSVC(max_iter=3000, random_state=42), cv=2)),
    ]
    return base

def fit_eval(feature_cols, tr, te, ext, drop=None, base_override=None):
    cols = [c for c in feature_cols if c != drop] if drop else feature_cols
    Xtr_raw, ytr = tr[cols].values.astype(float), tr['target'].values
    Xte_raw, yte = te[cols].values.astype(float), te['target'].values
    Xext_raw, yext = ext[cols].values.astype(float), ext['target'].values
    imp = SimpleImputer(strategy='median').fit(Xtr_raw)
    sc = StandardScaler().fit(imp.transform(Xtr_raw))
    prep = lambda X: sc.transform(imp.transform(X))
    Xtr, Xte, Xext = prep(Xtr_raw), prep(Xte_raw), prep(Xext_raw)
    ests = base_override if base_override else make_stack()
    stack = StackingClassifier(estimators=ests, final_estimator=LogisticRegression(max_iter=1000),
                                cv=StratifiedKFold(3, shuffle=True, random_state=42),
                                stack_method='predict_proba', n_jobs=-1)
    stack.fit(Xtr, ytr)
    auc_int = roc_auc_score(yte, stack.predict_proba(Xte)[:, 1])
    auc_ext = roc_auc_score(yext, stack.predict_proba(Xext)[:, 1])
    return auc_int, auc_ext

rows = []

# ---- (1) Effect of deduplication-before-split ----
raw_cardio = pd.read_csv('cardio_data/cardio_train.csv', sep=None, engine='python')
c = raw_cardio.copy()
implausible = (c['ap_hi'] <= 0) | (c['ap_lo'] <= 0) | (c['ap_hi'] < c['ap_lo']) | (c['ap_hi'] > 250) | \
              (c['ap_lo'] > 200) | (c['height'] < 100) | (c['height'] > 220) | (c['weight'] < 30) | (c['weight'] > 200)
c = c.loc[~implausible].reset_index(drop=True)
h = pd.DataFrame()
h['age_years'] = c['age']/365.25; h['sex_male'] = (c['gender']==2).astype(int)
h['bmi'] = c['weight']/((c['height']/100)**2); h['sys_bp']=c['ap_hi'].astype(float); h['dia_bp']=c['ap_lo'].astype(float)
h['smoker']=c['smoke'].astype(int); h['chol_cat']=c['cholesterol'].astype(int); h['gluc_cat']=c['gluc'].astype(int)
h['target']=c['cardio'].astype(int)
tr_wd, te_wd = train_test_split(h, test_size=0.2, random_state=42, stratify=h['target'])
tr_wd, _ = train_test_split(tr_wd, train_size=min(20000, len(tr_wd)), random_state=42, stratify=tr_wd['target'])
auc_int_wd, auc_ext_wd = fit_eval(CORE, tr_wd, te_wd, ext_df)
rows.append({'ablation':'WITH duplicates (no dedup-before-split)', 'internal_auc': auc_int_wd, 'external_auc': auc_ext_wd})

auc_int_nd, auc_ext_nd = fit_eval(CORE, train_df, test_df, ext_df)
rows.append({'ablation':'WITHOUT duplicates (dedup-before-split, proposed)', 'internal_auc': auc_int_nd, 'external_auc': auc_ext_nd})

# ---- (2) Stacking component ablation: remove one base learner at a time ----
full_names = ['LogisticRegression','RandomForest','XGBoost','LinearSVM']
full_est = make_stack()
auc_int_full, auc_ext_full = fit_eval(CORE, train_df, test_df, ext_df, base_override=full_est)
rows.append({'ablation':'Full stack (LR+RF+XGB+SVM)', 'internal_auc': auc_int_full, 'external_auc': auc_ext_full})

for drop_name in full_names:
    sub_est = [(n, m) for n, m in make_stack() if n != drop_name]
    ai, ae = fit_eval(CORE, train_df, test_df, ext_df, base_override=sub_est)
    rows.append({'ablation': f'Stack minus {drop_name}', 'internal_auc': ai, 'external_auc': ae})

# ---- (3) Feature-group ablation ----
groups = {
    'w/o age': 'age_years', 'w/o sex': 'sex_male', 'w/o BMI': 'bmi',
    'w/o systolic BP': 'sys_bp', 'w/o diastolic BP': 'dia_bp',
    'w/o smoking status': 'smoker', 'w/o cholesterol category': 'chol_cat', 'w/o glucose category': 'gluc_cat'
}
for label, drop_col in groups.items():
    ai, ae = fit_eval(CORE, train_df, test_df, ext_df, drop=drop_col)
    rows.append({'ablation': label, 'internal_auc': ai, 'external_auc': ae})

df = pd.DataFrame(rows)
df.to_csv(f'{OUT}/tables/ablation_results.csv', index=False)
print(df.to_string(index=False))

                                         ablation  internal_auc  external_auc
          WITH duplicates (no dedup-before-split)      0.792527      0.663458
WITHOUT duplicates (dedup-before-split, proposed)      0.796195      0.659645
                       Full stack (LR+RF+XGB+SVM)      0.796195      0.659645
                   Stack minus LogisticRegression      0.796195      0.659372
                         Stack minus RandomForest      0.795351      0.658078
                              Stack minus XGBoost      0.795533      0.668075
                            Stack minus LinearSVM      0.796196      0.659309
                                          w/o age      0.773259      0.642484
                                          w/o sex      0.795980      0.657542
                                          w/o BMI      0.794604      0.655328
                                  w/o systolic BP      0.763866      0.646096
                                 w/o diastolic BP      0.795183 

## 5. Explainable AI: SHAP, LIME, ELI5 + Kendall's τ Agreement

Computes global feature-importance rankings for the XGBoost base learner using three independent
explanation methods — SHAP (`TreeExplainer`, exact), LIME (local surrogate models aggregated over
60 sampled instances), and ELI5 (permutation importance, 15 shuffles) — then quantifies pairwise
agreement between the resulting rankings with Kendall's τ rank-correlation coefficient.

**Note:** this cell can take a couple of minutes to run (LIME's per-instance explanations dominate the
runtime).

In [ ]:
import pandas as pd, numpy as np, json, joblib
from scipy.stats import kendalltau
import shap
from lime.lime_tabular import LimeTabularExplainer
from eli5.sklearn import PermutationImportance

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

train_df = pd.read_csv(f'{OUT}/tables/cardio_train_split.csv')
test_df  = pd.read_csv(f'{OUT}/tables/cardio_test_split.csv')
prep = joblib.load(f'{OUT}/tables/preprocessor.joblib')
imputer, scaler = prep['imputer'], prep['scaler']
base_models = joblib.load(f'{OUT}/tables/base_models.joblib')
xgb_model = base_models['XGBoost']

Xtr = scaler.transform(imputer.transform(train_df[CORE].values.astype(float)))
Xte = scaler.transform(imputer.transform(test_df[CORE].values.astype(float)))

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(Xte), size=300, replace=False)
Xte_sample = Xte[sample_idx]

# ---------------- SHAP ----------------
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(Xte_sample)
shap_importance = np.abs(shap_values).mean(axis=0)
shap_rank = pd.Series(shap_importance, index=CORE).rank(ascending=False)

np.save(f'{OUT}/tables/shap_values_sample.npy', shap_values)
np.save(f'{OUT}/tables/shap_sample_X.npy', Xte_sample)

# ---------------- LIME ----------------
lime_explainer = LimeTabularExplainer(
    training_data=Xtr, feature_names=CORE, class_names=['no_cvd', 'cvd'],
    mode='classification', discretize_continuous=True, random_state=42
)

lime_importances = np.zeros(len(CORE))
n_lime = 60  # local explanations aggregated -> global importance (kept modest for runtime)
lime_sample_idx = rng.choice(len(Xte), size=n_lime, replace=False)
for i in lime_sample_idx:
    exp = lime_explainer.explain_instance(Xte[i], xgb_model.predict_proba, num_features=len(CORE))
    weights = dict(exp.as_map()[1])  # {feature_index: weight}
    for fidx, w in weights.items():
        lime_importances[fidx] += abs(w)
lime_importances /= n_lime
lime_rank = pd.Series(lime_importances, index=CORE).rank(ascending=False)

# ---------------- ELI5 (permutation importance) ----------------
perm = PermutationImportance(xgb_model, scoring='roc_auc', n_iter=15, random_state=42)
perm.fit(Xte, test_df['target'].values)
eli5_importance = perm.feature_importances_
eli5_rank = pd.Series(eli5_importance, index=CORE).rank(ascending=False)

# ---------------- Aggregate + Kendall's tau agreement ----------------
imp_df = pd.DataFrame({
    'feature': CORE,
    'shap_mean_abs': shap_importance,
    'shap_rank': shap_rank.values,
    'lime_mean_abs_weight': lime_importances,
    'lime_rank': lime_rank.values,
    'eli5_perm_importance': eli5_importance,
    'eli5_rank': eli5_rank.values,
}).sort_values('shap_rank')
imp_df.to_csv(f'{OUT}/tables/xai_feature_importance.csv', index=False)
print(imp_df.to_string(index=False))

tau_shap_lime, p_shap_lime = kendalltau(shap_rank, lime_rank)
tau_shap_eli5, p_shap_eli5 = kendalltau(shap_rank, eli5_rank)
tau_lime_eli5, p_lime_eli5 = kendalltau(lime_rank, eli5_rank)

agreement = {
    'SHAP_vs_LIME': {'kendall_tau': float(tau_shap_lime), 'p_value': float(p_shap_lime)},
    'SHAP_vs_ELI5': {'kendall_tau': float(tau_shap_eli5), 'p_value': float(p_shap_eli5)},
    'LIME_vs_ELI5': {'kendall_tau': float(tau_lime_eli5), 'p_value': float(p_lime_eli5)},
}
with open(f'{OUT}/tables/xai_agreement.json', 'w') as f:
    json.dump(agreement, f, indent=2)
print(json.dumps(agreement, indent=2))

  feature  shap_mean_abs  shap_rank  lime_mean_abs_weight  lime_rank  eli5_perm_importance  eli5_rank
   sys_bp       0.775012        1.0              0.301098        1.0              0.175237        1.0
age_years       0.281922        2.0              0.071311        2.0              0.038704        2.0
 chol_cat       0.237093        3.0              0.051652        4.0              0.030874        3.0
      bmi       0.116502        4.0              0.033497        6.0              0.005821        4.0
   dia_bp       0.093030        5.0              0.053554        3.0              0.004208        5.0
 gluc_cat       0.043226        6.0              0.012206        7.0              0.001869        6.0
   smoker       0.034985        7.0              0.041099        5.0              0.001485        7.0
 sex_male       0.018930        8.0              0.008285        8.0              0.000639        8.0
{
  "SHAP_vs_LIME": {
    "kendall_tau": 0.7142857142857142,
    "p_value": 0.0141

## 5b. Explainability for the FINAL Stacking Ensemble (Reviewer Comment C5)
The tri-method explainability study above (Section 5) was run on the XGBoost base learner only, which creates a mismatch with the deployed model (the stacking ensemble). This section repeats the SHAP / LIME / ELI5 analysis directly on the fitted `StackingClassifier`, using model-agnostic explainers (SHAP's permutation-based `Explainer`, LIME, and ELI5 permutation importance all treat the model as a black box via `predict_proba`), and reports whether the stack's explanation agreement and feature ranking differ from the XGBoost-only analysis.

In [ ]:
# ============================================================
# 5b. Explainability Agreement for the FINAL STACKING ENSEMBLE (addresses reviewer comment C5)
# ------------------------------------------------------------
# Section 5 explains only the XGBoost base learner. Reviewers correctly noted
# this creates a mismatch with the "tri-method explainability study" claim,
# since the deployed model is the stacking ensemble, not XGBoost alone. Here
# we repeat the SHAP/LIME/ELI5 analysis directly on the fitted
# StackingClassifier, using model-agnostic explainers (SHAP's permutation-
# based Explainer, LIME, and ELI5 permutation importance all treat the model
# as a black box via predict_proba, so they apply to the heterogeneous stack
# without modification). Sample sizes are kept smaller than Section 5, since
# black-box SHAP/LIME on a 4-learner stack is considerably slower than
# TreeExplainer on XGBoost alone.
# ============================================================
import pandas as pd, numpy as np, json, joblib
from scipy.stats import kendalltau
import shap
from lime.lime_tabular import LimeTabularExplainer
from eli5.sklearn import PermutationImportance

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

train_df = pd.read_csv(f'{OUT}/tables/cardio_train_split.csv')
test_df  = pd.read_csv(f'{OUT}/tables/cardio_test_split.csv')
prep = joblib.load(f'{OUT}/tables/preprocessor.joblib')
imputer, scaler = prep['imputer'], prep['scaler']
stack_model = joblib.load(f'{OUT}/tables/stack_model.joblib')

Xtr = scaler.transform(imputer.transform(train_df[CORE].values.astype(float)))
Xte = scaler.transform(imputer.transform(test_df[CORE].values.astype(float)))

rng = np.random.RandomState(42)
# Smaller samples than Section 5: black-box SHAP/LIME on the 4-learner stack
# is much more expensive per call than TreeExplainer on XGBoost alone.
bg_idx     = rng.choice(len(Xtr), size=100, replace=False)   # SHAP background
sample_idx = rng.choice(len(Xte), size=100, replace=False)   # SHAP explain set
Xtr_bg      = Xtr[bg_idx]
Xte_sample  = Xte[sample_idx]

# ---------------- SHAP (model-agnostic, on the stack's predict_proba) ----------------
stack_explainer = shap.Explainer(stack_model.predict_proba, Xtr_bg, seed=42)
stack_shap_out = stack_explainer(Xte_sample)
# class-1 (CVD) SHAP values
stack_shap_values = stack_shap_out.values[..., 1] if stack_shap_out.values.ndim == 3 else stack_shap_out.values
stack_shap_importance = np.abs(stack_shap_values).mean(axis=0)
stack_shap_rank = pd.Series(stack_shap_importance, index=CORE).rank(ascending=False)

np.save(f'{OUT}/tables/stack_shap_values_sample.npy', stack_shap_values)
np.save(f'{OUT}/tables/stack_shap_sample_X.npy', Xte_sample)

# ---------------- LIME (on stack.predict_proba) ----------------
lime_explainer_stack = LimeTabularExplainer(
    training_data=Xtr, feature_names=CORE, class_names=['no_cvd', 'cvd'],
    mode='classification', discretize_continuous=True, random_state=42
)
n_lime = 40  # kept modest: each call re-fits a local surrogate against the full 4-model stack
lime_importances_stack = np.zeros(len(CORE))
lime_sample_idx = rng.choice(len(Xte), size=n_lime, replace=False)
for i in lime_sample_idx:
    exp = lime_explainer_stack.explain_instance(Xte[i], stack_model.predict_proba, num_features=len(CORE))
    weights = dict(exp.as_map()[1])
    for fidx, w in weights.items():
        lime_importances_stack[fidx] += abs(w)
lime_importances_stack /= n_lime
lime_rank_stack = pd.Series(lime_importances_stack, index=CORE).rank(ascending=False)

# ---------------- ELI5 permutation importance (on the stack) ----------------
perm_stack = PermutationImportance(stack_model, scoring='roc_auc', n_iter=15, random_state=42)
perm_stack.fit(Xte, test_df['target'].values)
eli5_importance_stack = perm_stack.feature_importances_
eli5_rank_stack = pd.Series(eli5_importance_stack, index=CORE).rank(ascending=False)

# ---------------- Aggregate + Kendall's tau agreement (within the stack) ----------------
imp_df_stack = pd.DataFrame({
    'feature': CORE,
    'shap_mean_abs': stack_shap_importance, 'shap_rank': stack_shap_rank.values,
    'lime_mean_abs_weight': lime_importances_stack, 'lime_rank': lime_rank_stack.values,
    'eli5_perm_importance': eli5_importance_stack, 'eli5_rank': eli5_rank_stack.values,
}).sort_values('shap_rank')
imp_df_stack.to_csv(f'{OUT}/tables/xai_feature_importance_STACK.csv', index=False)
print('=== XAI feature importance: FINAL STACKING ENSEMBLE ===')
print(imp_df_stack.to_string(index=False))

tau_s_l, p_s_l = kendalltau(stack_shap_rank, lime_rank_stack)
tau_s_e, p_s_e = kendalltau(stack_shap_rank, eli5_rank_stack)
tau_l_e, p_l_e = kendalltau(lime_rank_stack, eli5_rank_stack)

agreement_stack = {
    'SHAP_vs_LIME': {'kendall_tau': float(tau_s_l), 'p_value': float(p_s_l)},
    'SHAP_vs_ELI5': {'kendall_tau': float(tau_s_e), 'p_value': float(p_s_e)},
    'LIME_vs_ELI5': {'kendall_tau': float(tau_l_e), 'p_value': float(p_l_e)},
}
with open(f'{OUT}/tables/xai_agreement_STACK.json', 'w') as f:
    json.dump(agreement_stack, f, indent=2)
print(json.dumps(agreement_stack, indent=2))

# ---------------- Cross-check: does the STACK's ranking match the XGBoost-only ranking from Section 5? ----------------
xgb_imp = pd.read_csv(f'{OUT}/tables/xai_feature_importance.csv').set_index('feature')
tau_xgb_vs_stack_shap, p_xgb_vs_stack_shap = kendalltau(
    xgb_imp.loc[CORE, 'shap_rank'].values, stack_shap_rank.values
)
print(f"\nKendall tau, XGBoost-only SHAP rank vs. Stacking-ensemble SHAP rank: "
      f"tau={tau_xgb_vs_stack_shap:.3f}, p={p_xgb_vs_stack_shap:.4f}")

with open(f'{OUT}/tables/xai_agreement_STACK.json', 'r') as f:
    d_ = json.load(f)
d_['XGBoost_base_learner_vs_Stack_SHAP'] = {
    'kendall_tau': float(tau_xgb_vs_stack_shap), 'p_value': float(p_xgb_vs_stack_shap)
}
with open(f'{OUT}/tables/xai_agreement_STACK.json', 'w') as f:
    json.dump(d_, f, indent=2)


ExactExplainer explainer: 101it [00:59,  1.55it/s]


=== XAI feature importance: FINAL STACKING ENSEMBLE ===
  feature  shap_mean_abs  shap_rank  lime_mean_abs_weight  lime_rank  eli5_perm_importance  eli5_rank
   sys_bp       0.170648        1.0              0.329811        1.0              0.168962        1.0
age_years       0.058236        2.0              0.064802        3.0              0.037427        2.0
 chol_cat       0.052962        3.0              0.090193        2.0              0.029144        3.0
      bmi       0.021147        4.0              0.027131        6.0              0.005441        4.0
   dia_bp       0.019241        5.0              0.060812        4.0              0.005105        5.0
   smoker       0.008383        6.0              0.034683        5.0              0.001195        7.0
 gluc_cat       0.006851        7.0              0.008304        7.0              0.001832        6.0
 sex_male       0.002464        8.0              0.005365        8.0              0.000652        8.0
{
  "SHAP_vs_LIME": {
    

## 6. Statistical Significance (McNemar's Test) + Reliability Analysis

- **McNemar's test** (paired, prediction-level) comparing (a) naive vs. leakage-free preprocessing
  predictions on the internal test set, and (b) the stacking ensemble vs. its best single base learner,
  on both the internal and external test sets.
- **Reliability analysis**: 1,000-iteration bootstrap 95% confidence intervals for AUC and Brier score
  (internal + external), expected calibration error (ECE), and 5-fold cross-validation fold-to-fold AUC
  stability for the diagnostic logistic-regression probe.

In [ ]:
import pandas as pd, numpy as np, json
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

d = np.load(f'{OUT}/tables/pred_arrays.npz')
yte, yext = d['yte'], d['yext']

def get(name, split, kind):
    return d[f'{kind}__{name}__{split}']

results = {}

# ---------------- (A) McNemar: Naive vs Leakage-free PREDICTIONS ----------------
# Reconstruct paired predictions for naive-preprocessing vs leakage-free-preprocessing
# pipelines on the SAME internal test set (Logistic Regression probe, as in s2).
h1 = pd.read_csv(f'{OUT}/tables/harmonized_cardio.csv')
X = h1[CORE].values.astype(float); y = h1['target'].values
Xtr, Xte, ytr, yte_lr = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# naive: impute+scale fit on FULL X (train+test together) before evaluating on this split
imp_naive = SimpleImputer(strategy='median').fit(X)
sc_naive = StandardScaler().fit(imp_naive.transform(X))
naive_pipe_clf = LogisticRegression(max_iter=1000).fit(sc_naive.transform(imp_naive.transform(Xtr)), ytr)
pred_naive = naive_pipe_clf.predict(sc_naive.transform(imp_naive.transform(Xte)))

# leakage-free: impute+scale fit on TRAIN ONLY
imp_clean = SimpleImputer(strategy='median').fit(Xtr)
sc_clean = StandardScaler().fit(imp_clean.transform(Xtr))
clean_clf = LogisticRegression(max_iter=1000).fit(sc_clean.transform(imp_clean.transform(Xtr)), ytr)
pred_clean = clean_clf.predict(sc_clean.transform(imp_clean.transform(Xte)))

correct_naive = (pred_naive == yte_lr)
correct_clean = (pred_clean == yte_lr)
# contingency table: [ [both correct, naive correct only], [clean correct only, both wrong] ]
b = int(np.sum(correct_naive & ~correct_clean))   # naive right, clean wrong
c_ = int(np.sum(~correct_naive & correct_clean))  # clean right, naive wrong
both_right = int(np.sum(correct_naive & correct_clean))
both_wrong = int(np.sum(~correct_naive & ~correct_clean))
table = [[both_right, b], [c_, both_wrong]]
mc1 = mcnemar(table, exact=(min(b, c_) < 25), correction=True)
results['mcnemar_naive_vs_leakagefree_preprocessing'] = {
    'contingency_table_[[both_correct,naive_only],[clean_only,both_wrong]]': table,
    'statistic': float(mc1.statistic), 'p_value': float(mc1.pvalue)
}

# ---------------- (B) McNemar: Stacking ensemble vs best single base learner ----------------
perf = pd.read_csv(f'{OUT}/tables/model_performance.csv')
best_base = perf[(perf.split=='internal_test') & (perf.model!='StackingEnsemble')].sort_values('auc', ascending=False).iloc[0]['model']

for split, ytrue in [('internal_test', yte), ('external_framingham', yext)]:
    pred_stack = get('StackingEnsemble', split, 'pred')
    pred_best = get(best_base, split, 'pred')
    corr_stack = (pred_stack == ytrue)
    corr_best = (pred_best == ytrue)
    b2 = int(np.sum(corr_stack & ~corr_best))
    c2 = int(np.sum(~corr_stack & corr_best))
    br = int(np.sum(corr_stack & corr_best)); bw = int(np.sum(~corr_stack & ~corr_best))
    tbl = [[br, b2], [c2, bw]]
    mc2 = mcnemar(tbl, exact=(min(b2, c2) < 25), correction=True)
    results[f'mcnemar_stacking_vs_{best_base}_{split}'] = {
        'contingency_table_[[both_correct,stack_only],[base_only,both_wrong]]': tbl,
        'statistic': float(mc2.statistic), 'p_value': float(mc2.pvalue)
    }

# ---------------- (C) Reliability analysis: bootstrap 95% CI for AUC/Accuracy ----------------
def bootstrap_ci(y_true, y_proba, n_boot=1000, metric='auc'):
    rng = np.random.RandomState(42)
    n = len(y_true)
    stats_ = []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        if len(np.unique(yt)) < 2:
            continue
        if metric == 'auc':
            stats_.append(roc_auc_score(yt, yp))
        elif metric == 'brier':
            stats_.append(brier_score_loss(yt, yp))
    lo, hi = np.percentile(stats_, [2.5, 97.5])
    return float(np.mean(stats_)), float(lo), float(hi)

def expected_calibration_error(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        m = (y_proba >= bins[i]) & (y_proba < bins[i+1] if i < n_bins-1 else y_proba <= bins[i+1])
        if m.sum() == 0: continue
        conf = y_proba[m].mean(); acc = y_true[m].mean()
        ece += (m.sum()/n) * abs(acc - conf)
    return float(ece)

rel = {}
for split, ytrue in [('internal_test', yte), ('external_framingham', yext)]:
    proba_stack = get('StackingEnsemble', split, 'proba')
    auc_mean, auc_lo, auc_hi = bootstrap_ci(ytrue, proba_stack, metric='auc')
    brier_mean, brier_lo, brier_hi = bootstrap_ci(ytrue, proba_stack, metric='brier')
    ece = expected_calibration_error(ytrue, proba_stack)
    rel[split] = {
        'auc_bootstrap_mean': auc_mean, 'auc_95ci': [auc_lo, auc_hi],
        'brier_bootstrap_mean': brier_mean, 'brier_95ci': [brier_lo, brier_hi],
        'expected_calibration_error': ece
    }

# Cross-validation stability (fold-to-fold variance) as a reliability indicator
Xh = h1[CORE].values.astype(float); yh = h1['target'].values
pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                  ('clf', LogisticRegression(max_iter=1000))])
skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof_proba = cross_val_predict(pipe, Xh, yh, cv=skf, method='predict_proba')[:, 1]
fold_aucs = []
for tr_i, te_i in skf.split(Xh, yh):
    fold_aucs.append(roc_auc_score(yh[te_i], oof_proba[te_i]))
rel['cv_fold_to_fold_stability_LR_probe'] = {
    'fold_aucs': fold_aucs, 'mean': float(np.mean(fold_aucs)), 'std': float(np.std(fold_aucs))
}

results['reliability_analysis'] = rel

with open(f'{OUT}/tables/stats_reliability.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(json.dumps(results, indent=2, default=str))

{
  "mcnemar_naive_vs_leakagefree_preprocessing": {
    "contingency_table_[[both_correct,naive_only],[clean_only,both_wrong]]": [
      [
        9906,
        0
      ],
      [
        0,
        3824
      ]
    ],
    "statistic": 0.0,
    "p_value": 1.0
  },
  "mcnemar_stacking_vs_XGBoost_internal_test": {
    "contingency_table_[[both_correct,stack_only],[base_only,both_wrong]]": [
      [
        9968,
        97
      ],
      [
        94,
        3571
      ]
    ],
    "statistic": 0.020942408376963352,
    "p_value": 0.8849359578757746
  },
  "mcnemar_stacking_vs_XGBoost_external_framingham": {
    "contingency_table_[[both_correct,stack_only],[base_only,both_wrong]]": [
      [
        1983,
        28
      ],
      [
        60,
        2169
      ]
    ],
    "statistic": 10.920454545454545,
    "p_value": 0.0009510827072544157
  },
  "reliability_analysis": {
    "internal_test": {
      "auc_bootstrap_mean": 0.7972625152339508,
      "auc_95ci": [
        0.790472874

## 6b. External Recalibration Experiment: Platt Scaling & Isotonic Regression (Reviewer Comment C8)
Section 6 showed severe external miscalibration (ECE $0.016 \to 0.419$) but did not test whether a lightweight, deployment-realistic fix could recover it. Here we split Framingham into a small calibration subset (30%) and a held-out external test set (70%), fit Platt scaling and isotonic regression on the calibration subset only, and re-evaluate ECE / AUC / Brier on the held-out external test set. Recalibration cannot change ranking (AUC is invariant to monotonic score transforms), so the ECE comparison is the relevant result here, not the AUC.

In [ ]:
# ============================================================
# 6b. External Recalibration Experiment: Platt Scaling & Isotonic Regression
# (addresses reviewer comment C8)
# ------------------------------------------------------------
# Section 6 showed severe external miscalibration (ECE 0.016 -> 0.419) but did
# not test whether a lightweight, deployment-realistic fix could recover it.
# Here we split Framingham into a small calibration subset (30%) and a
# held-out external test set (70%), fit Platt scaling (a 1D logistic
# regression on the stack's raw probabilities) and isotonic regression on
# the calibration subset ONLY, and re-evaluate ECE/AUC/Brier on the held-out
# external test set. By design, recalibration cannot change ranking (AUC is
# invariant to any monotonic transform of the scores), so any AUC change here
# is measurement noise from the train/test re-split, not a modeling effect;
# the ECE comparison is the relevant result.
# ============================================================
import pandas as pd, numpy as np, json, joblib
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

np.random.seed(42)
OUT = 'outputs'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

ext_df = pd.read_csv(f'{OUT}/tables/harmonized_framingham.csv')
prep = joblib.load(f'{OUT}/tables/preprocessor.joblib')
imputer, scaler = prep['imputer'], prep['scaler']
stack_model = joblib.load(f'{OUT}/tables/stack_model.joblib')

Xext_raw, yext = ext_df[CORE].values.astype(float), ext_df['target'].values
Xext = scaler.transform(imputer.transform(Xext_raw))

# ---- Split Framingham: small calibration subset vs. held-out external test ----
idx_cal, idx_test = train_test_split(
    np.arange(len(yext)), test_size=0.70, stratify=yext, random_state=42
)
X_cal, y_cal = Xext[idx_cal], yext[idx_cal]
X_test, y_test = Xext[idx_test], yext[idx_test]
print(f'Calibration subset: n={len(y_cal)} (prevalence={y_cal.mean():.3f}) | '
      f'Held-out external test: n={len(y_test)} (prevalence={y_test.mean():.3f})')

raw_probs_cal  = stack_model.predict_proba(X_cal)[:, 1]
raw_probs_test = stack_model.predict_proba(X_test)[:, 1]

def expected_calibration_error(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(y_true)
    for i in range(n_bins):
        m = (y_proba >= bins[i]) & (y_proba < bins[i+1] if i < n_bins-1 else y_proba <= bins[i+1])
        if m.sum() == 0: continue
        conf = y_proba[m].mean(); acc = y_true[m].mean()
        ece += (m.sum()/n) * abs(acc - conf)
    return float(ece)

# ---- Platt scaling: 1D logistic regression fit on calibration subset ----
platt = LogisticRegression().fit(raw_probs_cal.reshape(-1, 1), y_cal)
probs_platt = platt.predict_proba(raw_probs_test.reshape(-1, 1))[:, 1]

# ---- Isotonic regression fit on calibration subset ----
iso = IsotonicRegression(out_of_bounds='clip').fit(raw_probs_cal, y_cal)
probs_iso = iso.predict(raw_probs_test)

recal_results = {}
for label, probs in [('uncalibrated_raw', raw_probs_test),
                      ('platt_scaling', probs_platt),
                      ('isotonic_regression', probs_iso)]:
    recal_results[label] = {
        'auc': float(roc_auc_score(y_test, probs)),
        'brier': float(brier_score_loss(y_test, probs)),
        'ece': expected_calibration_error(y_test, probs),
    }

print('\n=== External recalibration results (held-out Framingham test, n=%d) ===' % len(y_test))
print(pd.DataFrame(recal_results).T.to_string())

with open(f'{OUT}/tables/external_recalibration.json', 'w') as f:
    json.dump({
        'calibration_subset_n': int(len(y_cal)),
        'held_out_external_test_n': int(len(y_test)),
        'results': recal_results
    }, f, indent=2)

# reliability diagram data points for plotting later (Section 7 figures)
def reliability_curve(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    mean_pred, obs_freq = [], []
    for i in range(n_bins):
        m = (y_proba >= bins[i]) & (y_proba < bins[i+1] if i < n_bins-1 else y_proba <= bins[i+1])
        if m.sum() == 0: continue
        mean_pred.append(float(y_proba[m].mean()))
        obs_freq.append(float(y_true[m].mean()))
    return mean_pred, obs_freq

curves = {}
for label, probs in [('uncalibrated_raw', raw_probs_test),
                      ('platt_scaling', probs_platt),
                      ('isotonic_regression', probs_iso)]:
    mp, of = reliability_curve(y_test, probs)
    curves[label] = {'mean_predicted': mp, 'observed_frequency': of}
with open(f'{OUT}/tables/external_recalibration_curves.json', 'w') as f:
    json.dump(curves, f, indent=2)


Calibration subset: n=1272 (prevalence=0.152) | Held-out external test: n=2968 (prevalence=0.152)

=== External recalibration results (held-out Framingham test, n=2968) ===
                          auc     brier       ece
uncalibrated_raw     0.657081  0.343965  0.419759
platt_scaling        0.657081  0.124251  0.002279
isotonic_regression  0.652165  0.125439  0.025921


## 7. Figure Generation

Generates all seven publication figures used in the paper (ROC curves, calibration diagram, SHAP
beeswarm summary, cross-method XAI importance comparison, Kendall's τ heatmap, ablation bar charts,
and the naive-vs-leakage-free bias comparison) and saves them to `outputs/figures/`.

In [ ]:
import pandas as pd, numpy as np, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
import shap

plt.rcParams.update({'font.size': 8, 'font.family': 'serif'})
OUT = 'outputs'
FIG = f'{OUT}/figures'
CORE = ['age_years','sex_male','bmi','sys_bp','dia_bp','smoker','chol_cat','gluc_cat']

d = np.load(f'{OUT}/tables/pred_arrays.npz')
yte, yext = d['yte'], d['yext']
perf = pd.read_csv(f'{OUT}/tables/model_performance.csv')
models = ['LogisticRegression','RandomForest','XGBoost','LinearSVM','StackingEnsemble']
colors = {'LogisticRegression':'#1f77b4','RandomForest':'#2ca02c','XGBoost':'#ff7f0e','LinearSVM':'#9467bd','StackingEnsemble':'#d62728'}

# ---------- Fig 1: ROC curves internal + external (side by side) ----------
fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
for split, ytrue, ax, title in [('internal_test', yte, axes[0], '(a) Internal test (cardio, held-out)'),
                                  ('external_framingham', yext, axes[1], '(b) External validation (Framingham)')]:
    for m in models:
        proba = d[f'proba__{m}__{split}']
        fpr, tpr, _ = roc_curve(ytrue, proba)
        auc = perf[(perf.model==m)&(perf.split==split)]['auc'].values[0]
        ax.plot(fpr, tpr, label=f'{m} (AUC={auc:.3f})', color=colors[m], linewidth=1.2)
    ax.plot([0,1],[0,1],'k--',linewidth=0.6)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontsize=8)
    ax.legend(fontsize=5.5, loc='lower right')
plt.tight_layout()
plt.savefig(f'{FIG}/fig1_roc_curves.png', dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.close()

# ---------- Fig 2: Calibration curves internal vs external (Stacking model) ----------
fig, ax = plt.subplots(figsize=(3.4, 3.0))
for split, ytrue, style, lbl in [('internal_test', yte, '-o', 'Internal test'),
                                   ('external_framingham', yext, '-s', 'External (Framingham)')]:
    proba = d[f'proba__StackingEnsemble__{split}']
    bins = np.linspace(0,1,11)
    confs, accs = [], []
    for i in range(10):
        m = (proba>=bins[i]) & (proba < bins[i+1] if i<9 else proba<=bins[i+1])
        if m.sum()==0: continue
        confs.append(proba[m].mean()); accs.append(ytrue[m].mean())
    ax.plot(confs, accs, style, label=lbl, linewidth=1.2, markersize=4)
ax.plot([0,1],[0,1],'k--',linewidth=0.6,label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed frequency')
ax.set_title('Reliability (calibration) diagram\nStacking ensemble', fontsize=8)
ax.legend(fontsize=6)
plt.tight_layout()
plt.savefig(f'{FIG}/fig2_calibration.png', dpi=300)
plt.close()

# ---------- Fig 3: SHAP summary (beeswarm) ----------
shap_values = np.load(f'{OUT}/tables/shap_values_sample.npy')
Xsample = np.load(f'{OUT}/tables/shap_sample_X.npy')
fig = plt.figure(figsize=(4.2, 3.4))
shap.summary_plot(shap_values, Xsample, feature_names=CORE, show=False, plot_size=None)
plt.tight_layout()
plt.savefig(f'{FIG}/fig3_shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()

# ---------- Fig 4: XAI method agreement (grouped bar of normalized importances) ----------
imp = pd.read_csv(f'{OUT}/tables/xai_feature_importance.csv')
imp_norm = imp.copy()
for col in ['shap_mean_abs','lime_mean_abs_weight','eli5_perm_importance']:
    imp_norm[col] = imp_norm[col] / imp_norm[col].max()
imp_norm = imp_norm.sort_values('shap_rank')
x = np.arange(len(imp_norm)); w = 0.25
fig, ax = plt.subplots(figsize=(4.6, 3.0))
ax.bar(x-w, imp_norm['shap_mean_abs'], width=w, label='SHAP', color='#d62728')
ax.bar(x, imp_norm['lime_mean_abs_weight'], width=w, label='LIME', color='#1f77b4')
ax.bar(x+w, imp_norm['eli5_perm_importance'], width=w, label='ELI5 (permutation)', color='#2ca02c')
ax.set_xticks(x); ax.set_xticklabels(imp_norm['feature'], rotation=40, ha='right', fontsize=6.5)
ax.set_ylabel('Normalized importance (max=1)')
ax.set_title('Cross-method feature-importance comparison', fontsize=8)
ax.legend(fontsize=6.5)
plt.tight_layout()
plt.savefig(f'{FIG}/fig4_xai_comparison.png', dpi=300)
plt.close()

# ---------- Fig 5: Kendall's tau agreement heatmap ----------
agree = json.load(open(f'{OUT}/tables/xai_agreement.json'))
methods = ['SHAP','LIME','ELI5']
mat = np.array([
    [1.0, agree['SHAP_vs_LIME']['kendall_tau'], agree['SHAP_vs_ELI5']['kendall_tau']],
    [agree['SHAP_vs_LIME']['kendall_tau'], 1.0, agree['LIME_vs_ELI5']['kendall_tau']],
    [agree['SHAP_vs_ELI5']['kendall_tau'], agree['LIME_vs_ELI5']['kendall_tau'], 1.0],
])
fig, ax = plt.subplots(figsize=(2.6, 2.4))
im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(methods, fontsize=7)
ax.set_yticks(range(3)); ax.set_yticklabels(methods, fontsize=7)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=7,
                 color='white' if mat[i,j]>0.6 else 'black')
ax.set_title("Kendall's $\\tau$ agreement", fontsize=8)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(f'{FIG}/fig5_kendall_heatmap.png', dpi=300)
plt.close()

# ---------- Fig 6: Ablation study bar charts ----------
ab = pd.read_csv(f'{OUT}/tables/ablation_results.csv')
feat_ab = ab[ab['ablation'].str.startswith('w/o')].copy()
comp_ab = ab[ab['ablation'].str.contains('stack', case=False) | ab['ablation'].str.contains('duplicate')].copy()

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
axes[0].barh(feat_ab['ablation'], feat_ab['internal_auc'], color='#4c72b0', label='Internal test')
axes[0].barh(feat_ab['ablation'], feat_ab['external_auc'], color='#dd8452', alpha=0.55, label='External (Framingham)')
axes[0].set_xlabel('AUC'); axes[0].set_title('(a) Feature-group ablation', fontsize=8)
axes[0].legend(fontsize=6)
axes[0].tick_params(axis='y', labelsize=6.5)

axes[1].barh(comp_ab['ablation'], comp_ab['internal_auc'], color='#4c72b0', label='Internal test')
axes[1].barh(comp_ab['ablation'], comp_ab['external_auc'], color='#dd8452', alpha=0.55, label='External (Framingham)')
axes[1].set_xlabel('AUC'); axes[1].set_title('(b) Dedup / component ablation', fontsize=8)
axes[1].legend(fontsize=6)
axes[1].tick_params(axis='y', labelsize=6.5)
plt.tight_layout()
plt.savefig(f'{FIG}/fig6_ablation.png', dpi=300, bbox_inches='tight', pad_inches=0.15)
plt.close()

# ---------- Fig 7: Naive vs leakage-free preprocessing bias ----------
leak = json.load(open(f'{OUT}/tables/leakage_analysis.json'))
fig, ax = plt.subplots(figsize=(3.4, 2.8))
labels = ['Impute+Scale\n(fold CV, LR probe)', 'SMOTE resampling\n(RF probe)']
naive_vals = [leak['naive_vs_leakagefree_preprocessing_5fold_AUC']['naive_mean_auc'],
              leak['naive_vs_leakagefree_SMOTE_resampling_AUC']['naive_smote_before_split_auc']]
clean_vals = [leak['naive_vs_leakagefree_preprocessing_5fold_AUC']['leakage_free_mean_auc'],
              leak['naive_vs_leakagefree_SMOTE_resampling_AUC']['leakage_free_smote_after_split_auc']]
x = np.arange(2); w=0.32
ax.bar(x-w/2, naive_vals, width=w, label='Naive (leaky)', color='#c44e52')
ax.bar(x+w/2, clean_vals, width=w, label='Leakage-free', color='#55a868')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7)
ax.set_ylabel('AUC'); ax.set_ylim(0.7, 0.85)
ax.set_title('Naive vs. leakage-free pipelines', fontsize=8)
ax.legend(fontsize=6.5)
plt.tight_layout()
plt.savefig(f'{FIG}/fig7_leakage_bias.png', dpi=300)
plt.close()

print("All figures saved to", FIG)
import os
print(os.listdir(FIG))

/tmp/ipykernel_1384/1852248105.py:60: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_values, Xsample, feature_names=CORE, show=False, plot_size=None)


All figures saved to outputs/figures
['fig6_ablation.png', 'fig7_leakage_bias.png', 'fig2_calibration.png', 'fig4_xai_comparison.png', 'fig5_kendall_heatmap.png', 'fig3_shap_summary.png', 'fig1_roc_curves.png']


## Summary

At this point `outputs/tables/` contains every numeric result reported in the paper
(`model_performance.csv`, `ablation_results.csv`, `xai_feature_importance.csv`, `xai_agreement.json`,
`leakage_analysis.json`, `stats_reliability.json`, `harmonize_log.json`) and `outputs/figures/` contains
all seven figures (`fig1_roc_curves.png` … `fig7_leakage_bias.png`) referenced in the manuscript.

## Download your results

Run this once the pipeline (Sections 1–7 below) has finished. It zips the whole
`outputs/` folder (tables, JSON results, and all seven figures) and triggers a
browser download. You can also right-click any single file in the Colab file
browser (left sidebar) to download it individually.

In [ ]:
# ============================================================
# Zip and download all results
# ============================================================
import shutil
from google.colab import files

shutil.make_archive('outputs', 'zip', 'outputs')
print("Created outputs.zip — starting download...")
files.download('outputs.zip')

Created outputs.zip — starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>